In [1]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Charger le CSV
df = pd.read_csv("../data/dataset_selection_sans_leger.csv")

# Aperçu du dataset
print("Aperçu du dataset :")
print(df.head())

print(f"\nNombre total d'images : {len(df)}")

# Répartition des labels
class_counts = df['label'].value_counts()
print("\nRépartition des labels :")
print(class_counts)

Aperçu du dataset :
                                                path     label
0  /home/mathis/Memoire/data/glaucome/LAG/LAG/Tra...  glaucome
1  /home/mathis/Memoire/data/glaucome/LAG/LAG/Tra...  glaucome
2  /home/mathis/Memoire/data/glaucome/LAG/LAG/Tes...  glaucome
3  /home/mathis/Memoire/data/glaucome/ORIGA/ORIGA...  glaucome
4  /home/mathis/Memoire/data/glaucome/LAG/LAG/Tes...  glaucome

Nombre total d'images : 6208

Répartition des labels :
label
glaucome    1552
mda         1552
diabete     1552
normaux     1552
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df,
    test_size=0.1,
    stratify=df['label'],
    random_state=42
)

print("Tailles des splits :")
print(f"Train : {len(df_train)}")
print(f"Test : {len(df_test)}")

# Mapping label -> id
classes = sorted(df['label'].unique().tolist())
label_to_id = {c: i for i, c in enumerate(classes)}
print(f"\nClasses : {classes}")

Tailles des splits :
Train : 5587
Test : 621

Classes : ['diabete', 'glaucome', 'mda', 'normaux']


In [3]:
# Fonction pour enlever les bords noirs
def crop_black_border(img, thr=10, pad=10):
    """
    img: numpy array BGR ou grayscale
    thr: seuil (0-255)
    pad: marge ajoutée autour du crop
    """
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img
    
    mask = gray > thr
    if not mask.any():
        return img
    
    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    
    y0 = max(0, y0 - pad)
    x0 = max(0, x0 - pad)
    y1 = min(img.shape[0] - 1, y1 + pad)
    x1 = min(img.shape[1] - 1, x1 + pad)
    
    return img[y0:y1+1, x0:x1+1]

# Fonction pour charger et prétraiter une image
def load_image(path, size=(128, 128)):
    """Charge une image, enlève les bords noirs et redimensionne"""
    img = cv2.imread(path)
    if img is None:
        return None
    img = crop_black_border(img, thr=10, pad=10)
    img = cv2.resize(img, size)
    return img

print("Fonctions de prétraitement définies ✓")

Fonctions de prétraitement définies ✓


In [4]:
from skimage.feature import hog

def extract_hog_features(img):
    """Extrait les features HOG d'une image BGR"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        feature_vector=True
    )
    return features

def extract_pixel_features(img):
    """Extrait les pixels aplatis (pour PCA)"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return gray.flatten().astype(np.float32) / 255.0

print("Fonctions d'extraction de features définies ✓")
print(f"HOG sur image 128x128 donne {extract_hog_features(np.zeros((128, 128, 3), dtype=np.uint8)).shape[0]} features")
print(f"Pixels aplatis sur image 128x128 donne {128*128} features")

Fonctions d'extraction de features définies ✓
HOG sur image 128x128 donne 8100 features
Pixels aplatis sur image 128x128 donne 16384 features


In [5]:
# Extraire les features de toutes les images
def extract_all_features(df_subset, extract_func):
    """Extrait les features pour un dataframe donné"""
    features = []
    labels = []
    valid_indices = []
    
    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
        img = load_image(row['path'])
        if img is None:
            continue
        
        feat = extract_func(img)
        features.append(feat)
        labels.append(label_to_id[row['label']])
        valid_indices.append(idx)
    
    return np.array(features), np.array(labels), valid_indices

print("Extraction des features pixels (pour PCA)...")
X_train_pixels, y_train, _ = extract_all_features(df_train, extract_pixel_features)
X_test_pixels, y_test, _ = extract_all_features(df_test, extract_pixel_features)

print(f"\nShape X_train_pixels: {X_train_pixels.shape}")
print(f"Shape X_test_pixels: {X_test_pixels.shape}")

Extraction des features pixels (pour PCA)...


100%|██████████| 621/621 [00:28<00:00, 22.02it/s]


Shape X_train_pixels: (5587, 16384)
Shape X_test_pixels: (621, 16384)


In [6]:
print("Extraction des features HOG...")
X_train_hog, _, _ = extract_all_features(df_train, extract_hog_features)
X_test_hog, _, _ = extract_all_features(df_test, extract_hog_features)

print(f"\nShape X_train_hog: {X_train_hog.shape}")
print(f"Shape X_test_hog: {X_test_hog.shape}")

Extraction des features HOG...


100%|██████████| 621/621 [00:30<00:00, 20.39it/s]


Shape X_train_hog: (5587, 8100)
Shape X_test_hog: (621, 8100)


In [7]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardiser les pixels (PCA sera appliquée dans le Pipeline GridSearch)
scaler_pixels = StandardScaler()
X_train_pixels_scaled = scaler_pixels.fit_transform(X_train_pixels)
X_test_pixels_scaled = scaler_pixels.transform(X_test_pixels)

print(f"Shape X_train_pixels_scaled: {X_train_pixels_scaled.shape}")
print("PCA sera testée avec différents n_components dans le GridSearch")

Shape X_train_pixels_scaled: (5587, 16384)
PCA sera testée avec différents n_components dans le GridSearch


In [8]:
# Standardiser HOG
scaler_hog = StandardScaler()
X_train_hog_scaled = scaler_hog.fit_transform(X_train_hog)
X_test_hog_scaled = scaler_hog.transform(X_test_hog)

# Préparer le dataset combiné (pixels_scaled + hog_scaled) pour PCA+HOG
# PCA sera appliquée uniquement sur les colonnes pixels via ColumnTransformer
X_train_combined_raw = np.hstack([X_train_pixels_scaled, X_train_hog_scaled])
X_test_combined_raw = np.hstack([X_test_pixels_scaled, X_test_hog_scaled])

n_pixel_features = X_train_pixels_scaled.shape[1]
n_hog_features = X_train_hog_scaled.shape[1]

print(f"Shape X_train_hog_scaled: {X_train_hog_scaled.shape}")
print(f"Shape X_train_combined_raw: {X_train_combined_raw.shape}")
print(f"  - Colonnes pixels: 0 à {n_pixel_features - 1}")
print(f"  - Colonnes HOG: {n_pixel_features} à {n_pixel_features + n_hog_features - 1}")

Shape X_train_hog_scaled: (5587, 8100)
Shape X_train_combined_raw: (5587, 24484)
  - Colonnes pixels: 0 à 16383
  - Colonnes HOG: 16384 à 24483


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Paramètres RF communs
rf_params = {
    'n_estimators': [20,100, 200, 300],
    'max_depth': [10, 20, 30,40, None],
    'min_samples_split': [2, 5,7, 10],
    'min_samples_leaf': [1, 2, 4,7],
    'max_features': ['sqrt', 'log2']
}

# Valeurs de PCA à tester
pca_components = [30,50,75, 100,125,150]

print("Paramètres RF à tester:")
for key, values in rf_params.items():
    print(f"  {key}: {values}")
print(f"\nComposantes PCA à tester: {pca_components}")

Paramètres RF à tester:
  n_estimators: [20, 100, 200, 300]
  max_depth: [10, 20, 30, 40, None]
  min_samples_split: [2, 5, 7, 10]
  min_samples_leaf: [1, 2, 4, 7]
  max_features: ['sqrt', 'log2']

Composantes PCA à tester: [30, 50, 75, 100, 125, 150]


In [10]:
results = {}
from sklearn.model_selection import RandomizedSearchCV
# ============================================================
# 1) PCA seul: Pipeline(PCA -> RF)
# ============================================================
print("="*60)
print("RandomizedSearch pour: PCA (avec n_components dans la grille)")
print("="*60)

pipe_pca = Pipeline([
    ('pca', PCA(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42))
])

param_grid_pca = {
    'pca__n_components': pca_components,
    **{f'rf__{k}': v for k, v in rf_params.items()}
}

grid_pca = RandomizedSearchCV(pipe_pca, param_grid_pca, n_iter=300, cv=3, scoring='accuracy', n_jobs=2, verbose=1, random_state=42)
grid_pca.fit(X_train_pixels_scaled, y_train)

print(f"\nMeilleurs paramètres: {grid_pca.best_params_}")
print(f"Meilleur score CV: {grid_pca.best_score_:.4f}")
print(f"Meilleur n_components PCA: {grid_pca.best_params_['pca__n_components']}")

results['PCA'] = {
    'grid_search': grid_pca,
    'best_model': grid_pca.best_estimator_,
    'best_params': grid_pca.best_params_,
    'best_cv_score': grid_pca.best_score_,
    'X_test': X_test_pixels_scaled
}

RandomizedSearch pour: PCA (avec n_components dans la grille)
Fitting 3 folds for each of 300 candidates, totalling 900 fits

Meilleurs paramètres: {'rf__n_estimators': 300, 'rf__min_samples_split': 2, 'rf__min_samples_leaf': 1, 'rf__max_features': 'log2', 'rf__max_depth': 20, 'pca__n_components': 100}
Meilleur score CV: 0.7664
Meilleur n_components PCA: 100


In [11]:
# ============================================================
# 2) HOG seul: RF directement
# ============================================================
print("="*60)
print("RandomizedSearch pour: HOG")
print("="*60)

rf_hog = RandomForestClassifier(random_state=42)

grid_hog = RandomizedSearchCV(rf_hog, rf_params, n_iter=300, cv=3, scoring='accuracy', n_jobs=2, verbose=1, random_state=42)
grid_hog.fit(X_train_hog_scaled, y_train)

print(f"\nMeilleurs paramètres: {grid_hog.best_params_}")
print(f"Meilleur score CV: {grid_hog.best_score_:.4f}")

results['HOG'] = {
    'grid_search': grid_hog,
    'best_model': grid_hog.best_estimator_,
    'best_params': grid_hog.best_params_,
    'best_cv_score': grid_hog.best_score_,
    'X_test': X_test_hog_scaled
}

RandomizedSearch pour: HOG
Fitting 3 folds for each of 300 candidates, totalling 900 fits


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 3) PCA + HOG: ColumnTransformer(PCA sur pixels, passthrough HOG) -> RF
# ============================================================
print("="*60)
print("RandomizedSearch pour: PCA + HOG (avec n_components dans la grille)")
print("="*60)

pixel_cols = list(range(n_pixel_features))
hog_cols = list(range(n_pixel_features, n_pixel_features + n_hog_features))

pipe_combined = Pipeline([
    ('features', ColumnTransformer([
        ('pca', PCA(random_state=42), pixel_cols),
        ('hog', 'passthrough', hog_cols)
    ])),
    ('rf', RandomForestClassifier(random_state=42))
])

param_grid_combined = {
    'features__pca__n_components': pca_components,
    **{f'rf__{k}': v for k, v in rf_params.items()}
}

grid_combined = RandomizedSearchCV(pipe_combined, param_grid_combined, n_iter=300, cv=3, scoring='accuracy', n_jobs=2, verbose=1, random_state=42)
grid_combined.fit(X_train_combined_raw, y_train)

print(f"\nMeilleurs paramètres: {grid_combined.best_params_}")
print(f"Meilleur score CV: {grid_combined.best_score_:.4f}")
print(f"Meilleur n_components PCA: {grid_combined.best_params_['features__pca__n_components']}")

results['PCA + HOG'] = {
    'grid_search': grid_combined,
    'best_model': grid_combined.best_estimator_,
    'best_params': grid_combined.best_params_,
    'best_cv_score': grid_combined.best_score_,
    'X_test': X_test_combined_raw
}

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

print("\n" + "="*70)
print("ÉVALUATION SUR LE JEU DE TEST")
print("="*70)

comparison_data = []

for name, data in results.items():
    model = data['best_model']
    X_test = data['X_test']
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    bacc = balanced_accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    results[name]['test_accuracy'] = acc
    results[name]['test_balanced_accuracy'] = bacc
    results[name]['test_f1'] = f1
    results[name]['y_pred'] = y_pred

    comparison_data.append({
        'Approche': name,
        'CV Score': data['best_cv_score'],
        'Test Accuracy': acc,
        'Balanced Acc': bacc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1
    })

    print(f"\n--- {name} ---")
    print(f"Meilleurs paramètres: {data['best_params']}")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Balanced Accuracy: {bacc:.4f}")
    print(f"F1 Score (macro): {f1:.4f}")

comparison_df = pd.DataFrame(comparison_data).round(4)

print("\n" + "="*70)
print("TABLEAU COMPARATIF DES 3 APPROCHES")
print("="*70)
print(comparison_df.to_string(index=False))

best_approach = comparison_df.loc[comparison_df['Test Accuracy'].idxmax(), 'Approche']
print(f"\nMeilleure approche: {best_approach}")

In [ ]:
# Matrices de confusion pour chaque approche
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, data) in enumerate(results.items()):
    y_pred = data['y_pred']
    cm = confusion_matrix(y_test, y_pred, labels=list(range(len(classes))))
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=axes[idx], values_format='d', cmap='Blues', colorbar=False)
    axes[idx].set_title(f'{name}\nAccuracy: {data["test_accuracy"]:.4f}')
    axes[idx].set_xlabel('Prédit')
    axes[idx].set_ylabel('Vrai')
    plt.setp(axes[idx].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()


In [ ]:
# Rapport de classification détaillé pour la meilleure approche
print("\n" + "="*70)
print(f"RAPPORT DÉTAILLÉ POUR LA MEILLEURE APPROCHE: {best_approach}")
print("="*70)

best_data = results[best_approach]
print(f"\nMeilleurs paramètres trouvés:")
for param, value in best_data['best_params'].items():
    print(f"  {param}: {value}")

print("\nRapport de classification:")
print(classification_report(y_test, best_data['y_pred'], target_names=classes, zero_division=0))

In [ ]:
# Graphique comparatif des performances
metrics = ['CV Score', 'Test Accuracy', 'Balanced Acc', 'F1 Score']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

for i, approach in enumerate(['PCA', 'HOG', 'PCA + HOG']):
    row = comparison_df[comparison_df['Approche'] == approach].iloc[0]
    values = [row['CV Score'], row['Test Accuracy'], row['Balanced Acc'], row['F1 Score']]
    ax.bar(x + i*width, values, width, label=approach)

ax.set_ylabel('Score')
ax.set_title('Comparaison des 3 approches de features')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

for i, approach in enumerate(['PCA', 'HOG', 'PCA + HOG']):
    row = comparison_df[comparison_df['Approche'] == approach].iloc[0]
    values = [row['CV Score'], row['Test Accuracy'], row['Balanced Acc'], row['F1 Score']]
    for j, v in enumerate(values):
        ax.text(x[j] + i*width, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.tree import plot_tree
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

# Meilleurs paramètres connus (PCA + HOG)
BEST_PCA_N = 300
BEST_RF_PARAMS = dict(
    n_estimators=300,
    max_depth=30,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=42
)

# Reconstruire le pipeline directement avec les meilleurs paramètres
pixel_cols = list(range(X_train_pixels_scaled.shape[1]))
hog_cols   = list(range(X_train_pixels_scaled.shape[1],
                        X_train_pixels_scaled.shape[1] + X_train_hog_scaled.shape[1]))

best_pipe = Pipeline([
    ('features', ColumnTransformer([
        ('pca', PCA(n_components=BEST_PCA_N, random_state=42), pixel_cols),
        ('hog', 'passthrough', hog_cols)
    ])),
    ('rf', RandomForestClassifier(**BEST_RF_PARAMS))
])

print("Entraînement du meilleur modèle PCA + HOG...")
best_pipe.fit(X_train_combined_raw, y_train)
print("Entraînement terminé ✓")

# Noms des features
feature_names = (
    [f'PCA_{i}' for i in range(BEST_PCA_N)] +
    [f'HOG_{i}' for i in range(X_train_hog_scaled.shape[1])]
)

# Visualiser l'arbre #1 (profondeur limitée à 4 pour la lisibilité)
classes = ['diabete', 'glaucome', 'mda', 'normaux']
tree_to_plot = best_pipe.named_steps['rf'].estimators_[0]

fig, ax = plt.subplots(figsize=(40, 20))
plot_tree(
    tree_to_plot,
    max_depth=5,
    feature_names=feature_names,
    class_names=classes,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=True,
    proportion=False
)
ax.set_title(
    "Arbre #1 du Random Forest — meilleur modèle PCA + HOG\n"
    f"n_estimators=300, max_depth=30, max_features=sqrt  —  affichage limité à profondeur 4",
    fontsize=14
)
plt.tight_layout()
plt.show()